In [ ]:
import h5py
import numpy as np
import pandas as pd
import plotly.express as px

f = r"J:\ctgroup\Edward\DATA\VMI\20250303\Propylene Oxide 2W\0.h5"
n = 10_000_000
with h5py.File(f, 'r') as file:
    x = file['x'][:n]
    y = file['y'][:n]
    t = file['t'][:n]
    tot = file['tot'][:n]
    pixel_corr = file['pixel_corr'][:n]
df = pd.DataFrame({'x': x, 'y': y, 't': t, 'tot': tot, 'pixel_corr': pixel_corr})

In [ ]:
# for pixel_corr in df['pixel_corr'].unique():
#     dff=df[df['pixel_corr']==pixel_corr]
#     if len(dff)<10:
#         continue
#     hist,xe,ye=np.histogram2d(dff['x'],dff['y'],bins=256,range=[[0,256],[0,256]],weights=dff['tot'])
#     hist=hist.T
#     fig=px.imshow(
#             hist,
#             origin='lower',
#             height=800,
#             width=800,
#             aspect="equal",
#
#     )
#     fig.show()

fig = px.density_heatmap(
        df[(df['t'] < 400) & (df['t'] > 200) & (df['tot'] < 256)],
        x='t',
        y='tot',
        nbinsx=256,
        nbinsy=256,
        histnorm='probability',
        height=800,
        width=800,
)
mean_tot = df[(df['t'] < 400) & (df['t'] > 200) & (df['tot'] < 256)].groupby('tot')['t'].median().reset_index()
counts_tot = df[(df['t'] < 400) & (df['t'] > 200) & (df['tot'] < 256)].groupby('tot').size().reset_index(name='counts')
fig.add_scatter(
        x=mean_tot['t'],
        y=mean_tot['tot'],
        mode='lines',
        line=dict(color='red', width=2),
)
fig2 = px.scatter(
        mean_tot,
        x='tot',
        y='t',
        height=800,
        width=800,
)
import scipy

smoothed = scipy.interpolate.make_smoothing_spline(mean_tot['tot'], mean_tot['t'], w=np.sqrt(counts_tot['counts']),
                                                   lam=1e4)(mean_tot['tot'])
fig2.add_scatter(
        x=mean_tot['tot'],
        y=smoothed,
        mode='lines',
        line=dict(color='blue', width=2),
)
#

fixed = np.where(mean_tot['tot'] > 10, smoothed, mean_tot['t'])

fig2.add_scatter(
        x=mean_tot['tot'],
        y=fixed,
        mode='lines',
        line=dict(color='green', width=2),
)

fig2.show()

save_loc = r"C:\Users\mcman\DataspellProjects\vmi-analysis\Analysis\timewalk_correction.npy"
np.save(save_loc, fixed)

In [ ]:
px.histogram(
        df[(df['t'] < 400) & (df['t'] > 200) & (df['tot'] < 256)], x='t', nbins=256, histnorm='probability', height=800,
        width=800, log_y=True
).show()
df['t_corrected'] = df['t'] - fixed[np.minimum(df['tot'].astype(int), 255)]

px.histogram(
        df[(df['t'] < 400) & (df['t'] > 200) & (df['tot'] < 255)],
        x='t_corrected',
        nbins=256,
        histnorm='probability',
        height=800,
        width=800,
        log_y=True
)